In [ ]:
%config Completer.use_jedi = False
import torch
import torchvision
import torch.nn.functional as F
from torch import nn, optim
from torch.autograd import Variable
from torchvision import transforms, datasets
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from torch.utils.data import Dataset, random_split
import sys, os
from tqdm import tqdm
import re


In [ ]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device:',device)
print('number of devices:',torch.cuda.device_count())
print('device name:',torch.cuda.get_device_name(0))
print('current device:',torch.cuda.current_device())


In [ ]:
epochs = 100
batch_size = 1

n1 = 2000
n2 = 3760

depth = 8
filters = 32
inch = 1
outch = 1

# === Directory containing field DAS data (.bin) to be denoised ===
data_dir_apply = './data/apply/'
data_dir = [data_dir_apply]

load_model = 'best'  # set 'last' or 'best'
save_dir = './models/model_first/'
test_dir = './apply_out/'

os.system("mkdir -p %s" %test_dir)
os.system("mkdir -p %s" %save_dir)


In [ ]:
def input_files(file_name):
    fin = open(file_name,"rb")
    patch = np.fromfile(fin,dtype='float32')
    fin.close()
    return patch

In [ ]:
class DasDataset(Dataset):
    def __init__(self, data_dir, n1, n2, sample_ratio=1, preload=True, transform_data1=None, transform_label=None):

        self.n1 = n1
        self.n2 = n2
        self.preload = preload
        self.sample_ratio = sample_ratio

        self.transform_data1 = transform_data1
        self.transform_label = transform_label

        data1_list = glob.glob(data_dir[0]+'*.bin')
        label_list = glob.glob(data_dir[0]+'*.bin')

        self.data1_list = sorted(data1_list)
        self.label_list = sorted(label_list)

        if len(data1_list) != len(label_list):
            print("ERROR: The number of shot and mask file is different")
            sys.exit(1)


        if preload:
            ntrain = 0
            x_train = []
            y_train = []
            for ii in tqdm(range(len(data1_list))):
                patch1 = input_files(self.data1_list[ii])
                patch1 = list(patch1)

                patch2 = input_files(self.label_list[ii])
                patch2 = list(patch2)

                x_train.append(patch1)
                y_train.append(patch2)

                ntrain = ntrain + 1

                del patch1
                del patch2

            x_train = np.array(x_train)
            y_train = np.array(y_train)

            self.x_train = x_train.reshape(-1,1,n2,n1)
            self.y_train = y_train.reshape(-1,1,n2,n1)
            print("shape of train data:", self.x_train.shape,self.y_train.shape)
            print("ndata: %d" %ntrain)


    def __len__(self):
        return len(self.data1_list)


    def __getitem__(self, idx):
        if self.preload:
            data1 = self.x_train[idx]
            label = self.y_train[idx]
        else:
            data1 = input_files(self.data1_list[idx])
            label = input_files(self.label_list[idx])
            data1 = data1.reshape(1,self.n2,self.n1)
            label = label.reshape(1,self.n2,self.n1)

        if self.transform_data1:
            data1 = self.transform_data1(data1)
        if self.transform_label:
            label = self.transform_label(label)
        return data1,label


In [ ]:
transform_data1 = transforms.Compose([transforms.ToTensor()])
transform_label = transforms.Compose([transforms.ToTensor()])

In [ ]:
dataset = DasDataset(
        data_dir = data_dir,
        n1 = n1,
        n2 = n2,
        sample_ratio = 1,
        preload = True)


train_loader = torch.utils.data.DataLoader(
        dataset = dataset,
        batch_size = batch_size,
        num_workers = 10
        )


In [ ]:
class DnCNN(nn.Module):
    def __init__(self, depth, filters, inch, outch):
        super(DnCNN, self).__init__()
        self.depth = depth
        self.filters = filters
        self.inch = inch
        self.outch = outch
        
        
        def first_block(inch,filters):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=filters, kernel_size=3, padding=1)]
            layers += [nn.ReLU()]
            
            xx = nn.Sequential(*layers)
            return xx
            
                
        def conv_block(filters, depth):
            layers = []
            for ii in range(depth-2):
                layers += [nn.Conv2d(in_channels=filters, out_channels=filters, kernel_size=3, padding=1)]
                layers += [nn.BatchNorm2d(filters)]
                layers += [nn.ReLU()]
            
            xx = nn.Sequential(*layers)
            return xx
            
        
        def last_block(outch,filters):
            layers = []
            layers += [nn.Conv2d(in_channels=filters, out_channels=outch, kernel_size=3, padding=1)]
                        
            xx = nn.Sequential(*layers)
            return xx

        
        self.first_block = first_block(inch,filters)
        self.conv_block  = conv_block(filters,depth)
        self.last_block  = last_block(outch,filters)      

        
    def forward(self,indata):
        xx = self.first_block(indata)
        xx = self.conv_block(xx)
        xx = self.last_block(xx)
        xx = torch.sub(indata,xx)
        
        return xx

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        
        return de
    

In [ ]:
class Autoencoder_r(nn.Module):
    def __init__(self):
        super(Autoencoder_r, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        
        return de

In [ ]:
class Autoencoder_s(nn.Module):
    def __init__(self):
        super(Autoencoder_s, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        xx = torch.sub(x,de)
        
        return xx

In [ ]:
class Autoencoder_c(nn.Module):
    def __init__(self):
        super(Autoencoder_c, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        
        return de

In [ ]:
class pcc_loss(nn.Module):
    def __init__(self):
        super(pcc_loss, self).__init__()
        
    def forward(self, true, pred):
        xx = pred
        yy = true
        
        vx = xx - torch.mean(xx)
        vy = yy - torch.mean(yy)
        
        pcc_loss = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2)) * torch.sqrt(torch.sum(vy ** 2)))
        
        return 1-torch.absolute(pcc_loss)


In [ ]:
model_r = Autoencoder_r().to(device)
model_s = DnCNN(depth=depth, filters=filters, inch=inch, outch=outch).to(device)
model_c = Autoencoder_c().to(device)

optimizer_r = torch.optim.Adam(model_r.parameters(), lr=0.001)
optimizer_s = torch.optim.Adam(model_s.parameters(), lr=0.001)
optimizer_c = torch.optim.Adam(model_c.parameters(), lr=0.001)

criterion1 = nn.L1Loss()
criterion2 = nn.MSELoss()

criterion3 = nn.L1Loss()
criterion4 = pcc_loss()

criterion5 = nn.L1Loss()
criterion6 = nn.MSELoss()

print(model_r)
print(model_s)
print(model_c)


In [ ]:
def findLastCheckpoint(save_dir):
    file_list = glob.glob(os.path.join(save_dir,'model_r_*.pth.tar'))  # get name list of all .hdf5 files
    if file_list:
        epochs_exist = []
        for file_ in file_list:
            result = re.findall(".*model_r_(.*).pth.tar*",file_)
            epochs_exist.append(int(result[0]))
        initial_epoch=max(epochs_exist)   
    else:
        initial_epoch = 1 
    return initial_epoch


In [ ]:
def save_checkpoint(model, filename="my_checkpoint.pth.tar"):
    checkpoint = {
        "state_dict": model.state_dict(),
    }
    torch.save(checkpoint, filename)

In [ ]:
initial_epoch = findLastCheckpoint(save_dir=save_dir)
if initial_epoch > epochs:
    initial_epoch = epochs 
    
if initial_epoch > 1:
    if load_model == 'best':
        print('load best model')
        checkpoint = torch.load(os.path.join(save_dir,'best_model_r.pth.tar'))
        model_r.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'best_model_s.pth.tar'))
        model_s.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'best_model_c.pth.tar'))
        model_c.load_state_dict(checkpoint['state_dict'])
        
    elif load_model == 'last':
        print('resuming by loading epoch %04d' %initial_epoch)
        checkpoint = torch.load(os.path.join(save_dir,'model_r_%04d.pth.tar' %initial_epoch))
        model_r.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'model_s_%04d.pth.tar' %initial_epoch))
        model_s.load_state_dict(checkpoint['state_dict'])
      
        checkpoint = torch.load(os.path.join(save_dir,'model_c_%04d.pth.tar' %initial_epoch))
        model_c.load_state_dict(checkpoint['state_dict'])
      

In [ ]:
def show_results(model, data_loader, device):
    model.eval()

    dataiter = iter(data_loader)
    images, labels = next(dataiter)
    images = images.to(device)

    with torch.no_grad():
        preds = model(images)
        images = images.to('cpu')
        preds = preds.to('cpu')

        inputs = images[:5].numpy()
        labels = labels[:5].numpy()
        preds = preds[:5].detach().numpy()

    figure_width = 20
    figure_height = 20

    # Create subplots
    fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(figure_width, figure_height))

    noise = inputs - preds
    # Plot subplots
    for i in range(5):
        # Row 1: input
        inimage = inputs[i,0,:,:]
        inimage = inimage.transpose(1,0)

        laimage = labels[i,0,:,:]
        laimage = laimage.transpose(1,0)

        primage = preds[i,0,:,:]
        primage = primage.transpose(1,0)

        noimage = noise[i,0,:,:]
        noimage = noimage.transpose(1,0)

        axes[0, i].imshow(inimage, cmap='gray')
        axes[0, i].set_title(f'True - Plot {i+1}')
        axes[0, i].axis('off')

        # Row 2: label
        axes[1, i].imshow(laimage, cmap='gray')
        axes[1, i].set_title(f'Label - Plot {i+1}')
        axes[1, i].axis('off')

        # Row 3: prediction
        axes[2, i].imshow(primage, cmap='gray')
        axes[2, i].set_title(f'Pred - Plot {i+1}')
        axes[2, i].axis('off')

        # Row 4: residual noise (input - prediction)
        axes[3, i].imshow(noimage, cmap='gray')
        axes[3, i].set_title(f'Noise - Plot {i+1}')
        axes[3, i].axis('off')

        axes[0, i].set_aspect('auto')
        axes[1, i].set_aspect('auto')
        axes[2, i].set_aspect('auto')
        axes[3, i].set_aspect('auto')

    plt.tight_layout()
    plt.show()


In [ ]:
def write_results(model_r, model_s, model_c, data_loader, device, test_dir):
    model_r.eval()
    model_c.eval()
    model_s.eval()

    number = 0
    with torch.no_grad():
        for batch_idx, data in enumerate(data_loader):
            inputs, labels = data
            inputs = inputs.to(device)

            preds1 = model_r(inputs)
            preds2 = model_s(preds1)

            idx1 = torch.randperm(preds2.shape[2])
            preds2_s1 = preds2[:,:,idx1,:].detach()

            preds3 = model_c(preds2_s1)

            inputs = inputs.to('cpu')
            preds1 = preds1.to('cpu')

            preds2 = preds2.to('cpu')
            preds3 = preds3.to('cpu')

            np.savetxt(test_dir+"random_number_%03d.txt"%number, idx1, fmt='%d')


            for jj in range(inputs.shape[0]):
                print(number)
                xxx = np.float32(inputs[jj,0,:,:])
                yyy = np.float32(preds1[jj,0,:,:])
                zzz = np.float32(labels[jj,0,:,:])
                lll = np.float32(preds2[jj,0,:,:])
                ddd = np.float32(preds3[jj,0,:,:])


                fout1 = open(os.path.join(test_dir,"input_%06d.bin"%number),"wb")
                fout2 = open(os.path.join(test_dir,"pred_r_%06d.bin"%number),"wb")
                fout3 = open(os.path.join(test_dir,"label_%06d.bin"%number),"wb")
                fout4 = open(os.path.join(test_dir,"pred_s_%06d.bin"%number),"wb")
                fout5 = open(os.path.join(test_dir,"pred_c_%06d.bin"%number),"wb")

                xxx.tofile(fout1)
                yyy.tofile(fout2)
                zzz.tofile(fout3)
                lll.tofile(fout4)
                ddd.tofile(fout5)


                fout1.close()
                fout2.close()
                fout3.close()
                fout4.close()
                fout5.close()

                number = number + 1

    print("write_results:",number)


In [ ]:
write_results(model_r, model_s, model_c, train_loader, device, test_dir)
